# 🖐️ YOLO26n 손(Hand) 탐지 모델 학습 — Colab 버전

체스 게임 손 탐지 시스템을 위한 2단계 학습 파이프라인입니다.

1. **Stage 1**: Roboflow Universe 공개 손 탐지 데이터셋으로 1차 학습
2. **Stage 2**: 직접 촬영/라벨링한 커스텀 데이터셋(실제 체스판+웹캠 환경)으로 파인튜닝

## 시작하기 전에
- 상단 메뉴에서 **런타임 > 런타임 유형 변경 > GPU(T4 이상)** 선택하고 시작하세요.
- [Roboflow API Key](https://app.roboflow.com/settings/api)를 미리 발급받아두세요.


## 0. 환경 설정 (GPU 확인 + 패키지 설치)

In [1]:
!nvidia-smi


Thu Jul  9 07:28:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q ultralytics roboflow
print("설치 완료")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.3/260.3 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 124.7 MB/s eta 0:00:00
설치 완료


## 1. Google Drive 연결 (선택, 강력 권장)

Colab은 세션이 끊기면 로컬 저장 내용이 사라집니다. 학습된 가중치를 Drive에 백업하려면 아래 셀을 실행하세요.
(사용 안 하시려면 이 셀은 건너뛰어도 됩니다.)


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/hand_yolo26_project"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"결과 저장 위치: {SAVE_DIR}")


Mounted at /content/drive
결과 저장 위치: /content/drive/MyDrive/hand_yolo26_project


## 2. Roboflow API Key 입력

아래 셀 실행 시 입력창이 뜨면 API Key를 붙여넣으세요. (화면에 노출되지 않습니다)


In [ ]:
from getpass import getpass
import os

os.environ["ROBOFLOW_API_KEY"] = getpass("Roboflow API Key 입력: ")
print("API Key 설정 완료")


## 3. Stage 1 — 오픈 데이터셋 다운로드

[Roboflow Universe](https://universe.roboflow.com)에서 "hand detection"으로 검색해서 원하는 공개 데이터셋을 찾은 뒤,
데이터셋 페이지의 코드 스니펫 탭에서 아래 3개 값을 확인해 채워넣으세요.

```python
# 예시 코드 스니펫 (Roboflow 데이터셋 페이지에서 확인 가능)
from roboflow import Roboflow
rf = Roboflow(api_key="...")
project = rf.workspace("WORKSPACE_NAME").project("PROJECT_NAME")
version = project.version(VERSION_NUMBER)
```


In [ ]:
# ===== 아래 3개 값을 실제 오픈 데이터셋 정보로 교체하세요 =====
OPEN_WORKSPACE = "YOUR_OPEN_DATASET_WORKSPACE"
OPEN_PROJECT = "YOUR_OPEN_DATASET_PROJECT"
OPEN_VERSION = 1
# ================================================================

from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace(OPEN_WORKSPACE).project(OPEN_PROJECT)
version = project.version(OPEN_VERSION)

# YOLO26/YOLO11 등 ultralytics 계열은 모두 'yolov8' 포맷 문자열을 사용합니다.
open_dataset = version.download("yolov8", location="datasets/open_hand_dataset")

OPEN_DATA_YAML = f"{open_dataset.location}/data.yaml"
print("\n[완료] data.yaml 위치:", OPEN_DATA_YAML)




In [ ]:
#roboflow에서 YOLOv26n으로 한번 학습하고 가져온 오픈데이터셋
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="EGCZl56bBEUValaQokVh")
project = rf.workspace("-nawzq").project("hand-detector-byo3i-mrlrf")
version = project.version(1)
dataset = version.download("yolo26")

In [8]:
#roboflow에서 학습안된 오픈데이터셋
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="Ct46NUZe7fgPCJLCDzgc")
project = rf.workspace("handdetection-w0uid").project("hand-detector-byo3i")
version = project.version(4)
#dataset = version.download("yolo26")
# YOLO26/YOLO11 등 ultralytics 계열은 모두 'yolov8' 포맷 문자열을 사용합니다.
open_dataset = version.download("yolov8", location="datasets/open_hand_dataset")

OPEN_DATA_YAML = f"{open_dataset.location}/data.yaml"
print("\n[완료] data.yaml 위치:", OPEN_DATA_YAML)


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to datasets/open_hand_dataset in yolov8:: 100%|██████████| 3874/3874 [00:00<00:00, 6918.52it/s]


[완료] data.yaml 위치: /content/datasets/open_hand_dataset/data.yaml


### data.yaml 클래스 확인

파인튜닝 단계와 클래스 구성이 일치해야 하니, 먼저 어떤 클래스로 구성되어 있는지 확인하세요.


In [9]:
import yaml

with open(OPEN_DATA_YAML) as f:
    print(yaml.safe_load(f))


{'names': ['hands'], 'nc': 1, 'roboflow': {'license': 'Public Domain', 'project': 'hand-detector-byo3i', 'url': 'https://universe.roboflow.com/handdetection-w0uid/hand-detector-byo3i/dataset/4', 'version': 4, 'workspace': 'handdetection-w0uid'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}


## 4. Stage 1 — YOLO26n 1차 학습

COCO 사전학습된 `yolo26n.pt`에서 시작해 손 탐지를 학습합니다.


In [10]:
from ultralytics import YOLO

model_stage1 = YOLO("yolo26n.pt")

results_stage1 = model_stage1.train(
    data=OPEN_DATA_YAML,
    epochs=150,
    imgsz=640,
    batch=16,
    device=0,
    project="runs/stage1_open",
    name="yolo26n_hand_open",
    patience=20,
    exist_ok=True,
)

STAGE1_BEST = "runs/stage1_open/yolo26n_hand_open/weights/best.pt"
print("\n[완료] Stage1 best 가중치:", STAGE1_BEST)


Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/open_hand_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n_hand_open, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ov

In [14]:
import os
import shutil

SAVE_DIR = "/content/drive/MyDrive/hand_yolo26_project"

# 먼저 상태 확인
print("현재 위치:", os.getcwd())
print("runs/stage1_open 존재?:", os.path.exists("runs/stage1_open"))

if os.path.exists("/content/drive/MyDrive") and os.path.exists("runs/stage1_open"):
    os.makedirs(SAVE_DIR, exist_ok=True)
    shutil.copytree(
        "runs/stage1_open",
        f"{SAVE_DIR}/runs_stage1_open",
        dirs_exist_ok=True
    )
    print("Drive 백업 완료 ->", f"{SAVE_DIR}/runs_stage1_open")
else:
    print("백업 실패: Drive 미연결 또는 학습 결과 폴더 없음")

현재 위치: /content
runs/stage1_open 존재?: False
백업 실패: Drive 미연결 또는 학습 결과 폴더 없음


In [15]:
SAVE_DIR = "/content/drive/MyDrive/hand_yolo26_project"
os.makedirs(SAVE_DIR, exist_ok=True)

if os.path.exists("/content/drive/MyDrive") and os.path.exists("runs/stage1_open"):
    shutil.copytree("runs/stage1_open", f"{SAVE_DIR}/runs/stage1_open", dirs_exist_ok=True)
    STAGE1_BEST = f"{SAVE_DIR}/runs/stage1_open/yolo26n_hand_open/weights/best.pt"
    print("Drive 백업 완료 ->", STAGE1_BEST)

In [17]:
SAVE_DIR = "/content/drive/MyDrive/hand_yolo26_project"
os.makedirs(SAVE_DIR, exist_ok=True)

STAGE1_RESULT_DIR = "/content/runs/detect/runs/stage1_open/yolo26n_hand_open"

if os.path.exists("/content/drive/MyDrive") and os.path.exists(STAGE1_RESULT_DIR):
    shutil.copytree(
        STAGE1_RESULT_DIR,
        f"{SAVE_DIR}/runs_stage1_open",
        dirs_exist_ok=True
    )
    # Drive에 백업된 최종 경로로 STAGE1_BEST 갱신 (이후 Stage2에서 이 변수 사용)
    STAGE1_BEST = f"{SAVE_DIR}/runs_stage1_open/weights/best.pt"
    print("Drive 백업 완료 ->", STAGE1_BEST)
    print("백업 파일 존재?:", os.path.exists(STAGE1_BEST))
else:
    print("백업 실패: Drive 미연결 또는 결과 폴더 없음")

Drive 백업 완료 -> /content/drive/MyDrive/hand_yolo26_project/runs_stage1_open/weights/best.pt
백업 파일 존재?: True


In [20]:
import os
import shutil

# 실제 저장된 경로로 고정
STAGE1_BEST = "/content/runs/detect/runs/stage1_open/yolo26n_hand_open/weights/best.pt"

# 파일 존재 확인
print("파일 존재?:", os.path.exists(STAGE1_BEST))

# Drive로 백업
SAVE_DIR = "/content/drive/MyDrive/hand_yolo26_project"
os.makedirs(SAVE_DIR, exist_ok=True)

STAGE1_RESULT_DIR = "/content/runs/detect/runs/stage1_open/yolo26n_hand_open"

if os.path.exists("/content/drive/MyDrive") and os.path.exists(STAGE1_RESULT_DIR):
    shutil.copytree(
        STAGE1_RESULT_DIR,
        f"{SAVE_DIR}/runs_stage1_open",
        dirs_exist_ok=True
    )
    # Drive에 백업된 최종 경로로 STAGE1_BEST 갱신 (이후 Stage2에서 이 변수 사용)
    STAGE1_BEST = f"{SAVE_DIR}/runs_stage1_open/weights/best.pt"
    print("Drive 백업 완료 ->", STAGE1_BEST)
    print("백업 파일 존재?:", os.path.exists(STAGE1_BEST))
else:
    print("백업 실패: Drive 미연결 또는 결과 폴더 없음")

파일 존재?: True
Drive 백업 완료 -> /content/drive/MyDrive/hand_yolo26_project/runs_stage1_open/weights/best.pt
백업 파일 존재?: True


/content/runs/detect/runs/stage1_open/yolo26n_hand_open/runs_stage1_open/weights/best.pt

## 5. Stage 2 — 커스텀 데이터셋 다운로드

본인이 Roboflow에 업로드하고 라벨링한 프로젝트 정보를 입력하세요.
(실제 체스판+웹캠 환경에서 촬영한 손 이미지로 구성된 데이터셋)


In [ ]:
# ===== 아래 3개 값을 본인 커스텀 데이터셋 정보로 교체하세요 =====
CUSTOM_WORKSPACE = "YOUR_WORKSPACE"
CUSTOM_PROJECT = "YOUR_PROJECT"
CUSTOM_VERSION = 1
# ================================================================

custom_project = rf.workspace(CUSTOM_WORKSPACE).project(CUSTOM_PROJECT)
custom_version = custom_project.version(CUSTOM_VERSION)

custom_dataset = custom_version.download("yolov8", location="datasets/custom_hand_dataset")

CUSTOM_DATA_YAML = f"{custom_dataset.location}/data.yaml"
print("\n[완료] data.yaml 위치:", CUSTOM_DATA_YAML)

with open(CUSTOM_DATA_YAML) as f:
    print(yaml.safe_load(f))


In [ ]:
#커스텀 데이터셋 다운로드
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="Dt9J8o734uMCvC3iEsh8")
project = rf.workspace("shin-dongsoo").project("hand-detection-fsr0m")
version = project.version(8)
dataset = version.download("yolo26")


In [31]:
!pip install -q roboflow

from roboflow import Roboflow
import os
import yaml

rf = Roboflow(api_key="Dt9J8o734uMCvC3iEsh8")
project = rf.workspace("shin-dongsoo").project("hand-detection-fsr0m")
version = project.version(8)

# YOLO26 포맷으로 직접 다운로드
custom_dataset = version.download("yolo26", location="/content/datasets/custom_hand_dataset")

CUSTOM_DATA_YAML = f"{custom_dataset.location}/data.yaml"
print("\n[완료] data.yaml 위치:", CUSTOM_DATA_YAML)

with open(CUSTOM_DATA_YAML) as f:
    custom_yaml = yaml.safe_load(f)
    print(custom_yaml)

loading Roboflow workspace...
loading Roboflow project...

[완료] data.yaml 위치: /content/datasets/custom_hand_dataset/data.yaml
{'names': ['hand'], 'nc': 1, 'roboflow': {'license': 'CC BY 4.0', 'project': 'hand-detection-fsr0m', 'url': 'https://universe.roboflow.com/shin-dongsoo/hand-detection-fsr0m/dataset/8', 'version': 8, 'workspace': 'shin-dongsoo'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}


In [41]:
import os
import shutil
import random

def resplit_train_only_dataset(
    dataset_dir,
    train_ratio=0.7,
    valid_ratio=0.2,
    test_ratio=0.1,
    seed=42
):
    assert abs(train_ratio + valid_ratio + test_ratio - 1.0) < 1e-6, "비율 합은 1.0이어야 합니다"

    src_images = os.path.join(dataset_dir, "train", "images")
    src_labels = os.path.join(dataset_dir, "train", "labels")

    assert os.path.exists(src_images), f"이미지 폴더 없음: {src_images}"

    # 이미지 목록 확보 및 셔플
    random.seed(seed)
    images = [f for f in os.listdir(src_images) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(images)

    n = len(images)
    n_train = int(n * train_ratio)
    n_valid = int(n * valid_ratio)

    splits = {
        'train': images[:n_train],
        'valid': images[n_train:n_train + n_valid],
        'test': images[n_train + n_valid:],
    }

    # 임시 폴더에 새로 나눠 담기 (기존 train 폴더와 충돌 방지)
    tmp_dir = dataset_dir + "_resplit"

    for split_name, files in splits.items():
        img_out = os.path.join(tmp_dir, split_name, 'images')
        lbl_out = os.path.join(tmp_dir, split_name, 'labels')
        os.makedirs(img_out, exist_ok=True)
        os.makedirs(lbl_out, exist_ok=True)

        for fname in files:
            shutil.copy(os.path.join(src_images, fname), os.path.join(img_out, fname))
            label_name = os.path.splitext(fname)[0] + '.txt'
            label_path = os.path.join(src_labels, label_name)
            if os.path.exists(label_path):
                shutil.copy(label_path, os.path.join(lbl_out, label_name))

        print(f"{split_name}: {len(files)}장")

    # 기존 폴더를 새로 나눈 결과로 교체
    shutil.rmtree(dataset_dir)
    shutil.move(tmp_dir, dataset_dir)

    print(f"\n[완료] 재분할된 데이터셋 위치: {dataset_dir}")
    return dataset_dir


# 실행
CUSTOM_DATASET_DIR = "/content/datasets/custom_hand_dataset"
resplit_train_only_dataset(CUSTOM_DATASET_DIR, train_ratio=0.7, valid_ratio=0.2, test_ratio=0.1)

train: 125장
valid: 36장
test: 19장

[완료] 재분할된 데이터셋 위치: /content/datasets/custom_hand_dataset


'/content/datasets/custom_hand_dataset'

In [48]:
import yaml
import os

CUSTOM_DATASET_DIR = "/content/datasets/custom_hand_dataset"
data_yaml_path = os.path.join(CUSTOM_DATASET_DIR, "data.yaml")

new_cfg = {
    'train': '../train/images',
    'val': '../valid/images',
    'test': '../test/images',
    'nc': 1,
    'names': ['hand'],
}

with open(data_yaml_path, 'w') as f:
    yaml.dump(new_cfg, f)

print("data.yaml 새로 생성됨:", new_cfg)

data.yaml 새로 생성됨: {'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 1, 'names': ['hand']}


**클래스 확인 체크포인트**: 위 출력의 `names`가 Stage1 오픈 데이터셋의 `names`와 동일한지 (예: 둘 다 `['hand']`) 꼭 확인하세요. 다르면 파인튜닝 시 감지 헤드가 재구성되어 Stage1 학습 효과가 일부 손실될 수 있습니다.

## 6. Stage 2 — 커스텀 데이터셋으로 파인튜닝

Stage1 가중치를 초기값으로 불러와서, 학습률을 낮추고 실제 환경 데이터로 추가 학습합니다.
데이터가 적을수록 `epochs`를 낮게 잡아 과적합을 방지하세요 (기본값 50, 데이터가 200장 미만이면 20~30 권장).


In [49]:
with open(OPEN_DATA_YAML) as f:
    open_yaml = yaml.safe_load(f)

print("Stage1 (오픈) 클래스:", open_yaml['names'])
print("Stage2 (커스텀) 클래스:", custom_yaml['names'])

if open_yaml['names'] != custom_yaml['names']:
    print("⚠️ 클래스 구성이 다릅니다! 파인튜닝 시 감지 헤드가 재구성될 수 있어요.")
else:
    print("✅ 클래스 일치 확인 완료")

Stage1 (오픈) 클래스: ['hand']
Stage2 (커스텀) 클래스: ['hand']
✅ 클래스 일치 확인 완료


In [50]:
import yaml

def unify_class_name(yaml_path, target_name="hand"):
    with open(yaml_path) as f:
        data = yaml.safe_load(f)

    old_names = data['names']
    data['names'] = [target_name]  # 단일 클래스라고 가정하고 통일

    with open(yaml_path, 'w') as f:
        yaml.dump(data, f, allow_unicode=True)

    print(f"{yaml_path}")
    print(f"  변경 전: {old_names}")
    print(f"  변경 후: {data['names']}")

# Stage1 (오픈 데이터셋) 'hands' -> 'hand'
unify_class_name(OPEN_DATA_YAML, "hand")

print()

# Stage2 (커스텀 데이터셋) 이미 'hand'지만 확인차 동일하게 적용
unify_class_name(CUSTOM_DATA_YAML, "hand")

/content/datasets/open_hand_dataset/data.yaml
  변경 전: ['hand']
  변경 후: ['hand']

/content/datasets/custom_hand_dataset/data.yaml
  변경 전: ['hand']
  변경 후: ['hand']


In [51]:
with open(OPEN_DATA_YAML) as f:
    print("Stage1 (오픈) 클래스:", yaml.safe_load(f)['names'])

with open(CUSTOM_DATA_YAML) as f:
    print("Stage2 (커스텀) 클래스:", yaml.safe_load(f)['names'])

Stage1 (오픈) 클래스: ['hand']
Stage2 (커스텀) 클래스: ['hand']


In [52]:
model_stage2 = YOLO("/content/drive/MyDrive/hand_yolo26_project/runs_stage1_open/weights/best.pt")

results_stage2 = model_stage2.train(
    data=CUSTOM_DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    lr0=0.001,       # 기본 0.01보다 낮게 -> 파인튜닝에 적합
    device=0,
    project="runs/stage2_finetune",
    name="yolo26n_hand_finetuned",
    patience=15,
    exist_ok=True,
)

STAGE2_BEST = "runs/stage2_finetune/yolo26n_hand_finetuned/weights/best.pt"
print("\n[완료] 최종 가중치:", STAGE2_BEST)


Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/custom_hand_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/hand_yolo26_project/runs_stage1_open/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n_hand_fine

In [54]:
STAGE2_LOCAL_DIR = str(model_stage2.trainer.save_dir)
STAGE2_BEST_LOCAL = os.path.join(STAGE2_LOCAL_DIR, "weights", "best.pt")

In [55]:
SAVE_DIR = "/content/drive/MyDrive/hand_yolo26_project"
os.makedirs(SAVE_DIR, exist_ok=True)

DRIVE_TARGET = os.path.join(SAVE_DIR, "runs_stage2_finetune")

if os.path.exists("/content/drive/MyDrive") and os.path.exists(STAGE2_LOCAL_DIR):
    shutil.copytree(STAGE2_LOCAL_DIR, DRIVE_TARGET, dirs_exist_ok=True)
    STAGE2_BEST = os.path.join(DRIVE_TARGET, "weights", "best.pt")
    print("Drive 백업 완료 ->", STAGE2_BEST)
    print("파일 존재?:", os.path.exists(STAGE2_BEST))
else:
    print("백업 실패: Drive 미연결 또는 학습 결과 폴더 없음")

Drive 백업 완료 -> /content/drive/MyDrive/hand_yolo26_project/runs_stage2_finetune/weights/best.pt
파일 존재?: True


In [ ]:
# (선택) Drive에 최종 결과 백업 -- 실전 배포용 최종 모델
import shutil
if os.path.exists("/content/drive"):
    shutil.copytree("runs/stage2_finetune", f"{SAVE_DIR}/runs_stage2_finetune", dirs_exist_ok=True)
    shutil.copy(STAGE2_BEST, f"{SAVE_DIR}/hand_yolo26n_final.pt")
    print("Drive 백업 완료 ->", f"{SAVE_DIR}/hand_yolo26n_final.pt")


In [56]:
import os
import shutil
from ultralytics import YOLO

# Stage1 가중치 로드
model_stage2 = YOLO("/content/drive/MyDrive/hand_yolo26_project/runs_stage1_open/weights/best.pt")

results_stage2 = model_stage2.train(
    data=CUSTOM_DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    lr0=0.001,
    device=0,
    project="/content/runs/stage2_finetune",   # 절대경로로 명시 (경로 중첩 방지)
    name="yolo26n_hand_finetuned",
    patience=15,
    exist_ok=True,
)

# 하드코딩 대신 실제 저장 경로를 trainer에서 직접 가져옴
STAGE2_LOCAL_DIR = str(model_stage2.trainer.save_dir)
STAGE2_BEST_LOCAL = os.path.join(STAGE2_LOCAL_DIR, "weights", "best.pt")

print("\n[완료] 로컬 저장 경로:", STAGE2_BEST_LOCAL)
print("파일 존재?:", os.path.exists(STAGE2_BEST_LOCAL))

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/custom_hand_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/hand_yolo26_project/runs_stage1_open/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n_hand_fine

In [57]:
SAVE_DIR = "/content/drive/MyDrive/hand_yolo26_project"
os.makedirs(SAVE_DIR, exist_ok=True)

DRIVE_TARGET = os.path.join(SAVE_DIR, "runs_stage2_finetune")

if os.path.exists("/content/drive/MyDrive") and os.path.exists(STAGE2_LOCAL_DIR):
    shutil.copytree(STAGE2_LOCAL_DIR, DRIVE_TARGET, dirs_exist_ok=True)
    STAGE2_BEST = os.path.join(DRIVE_TARGET, "weights", "best.pt")
    print("Drive 백업 완료 ->", STAGE2_BEST)
    print("파일 존재?:", os.path.exists(STAGE2_BEST))
else:
    print("백업 실패: Drive 미연결 또는 학습 결과 폴더 없음")

Drive 백업 완료 -> /content/drive/MyDrive/hand_yolo26_project/runs_stage2_finetune/weights/best.pt
파일 존재?: True


## 7. 검증 (mAP 확인)

In [ ]:
metrics = model_stage2.val(data=CUSTOM_DATA_YAML)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)


## 8. 테스트 추론

이미지 한 장 업로드해서 최종 모델이 손을 잘 잡는지 확인해보세요.


In [ ]:
from google.colab import files
from IPython.display import Image as IPImage, display

uploaded = files.upload()  # 테스트 이미지 업로드
test_image = list(uploaded.keys())[0]

final_model = YOLO(STAGE2_BEST)
results = final_model.predict(source=test_image, conf=0.5, save=True)

for r in results:
    print(f"탐지된 손 개수: {len(r.boxes)}")

# 결과 이미지 표시
saved_path = results[0].save_dir + "/" + test_image
display(IPImage(filename=saved_path))


## 9. 최종 모델 다운로드

로컬 PC로 최종 가중치 파일을 다운로드합니다. (실제 체스 프로젝트 코드에서 `YOLO("hand_yolo26n_final.pt")`로 불러와 사용하세요)


In [ ]:
from google.colab import files

files.download(STAGE2_BEST)


---
## 다음 단계 참고

이 노트북에서 만든 `hand_yolo26n_final.pt`는 앞서 설계한 전체 파이프라인에서 **손 탐지 전용 모델**로 사용됩니다.

```
웹캠 프레임
    ├─→ [체스판+기물 탐지 모델 (YOLOv11s, 별도)] → 보드 캘리브레이션 + 기물 위치
    ├─→ [손 탐지 모델 (여기서 만든 YOLO26n)] → 턴 전환 트리거
    └─→ 상태 머신에서 결과 조합 (디바운싱 + 기물 이동 검증)
```

- 손 탐지는 5~10fps 정도로 실시간 반복 실행
- conf threshold는 실전 환경에서 0.4~0.6 사이 직접 튜닝 권장
- 체스판 탐지 모델은 카메라 고정 시 캘리브레이션 단계에서 1회만 실행해도 충분
